# AISStreamMock + AISKafkaPredictor Integration Test

Tests the full pipeline without a live Kafka broker:
1. `AISStreamMock` streams records from Parquet files (no sleeps via high `speed_factor`)
2. Messages are fed directly into `AISKafkaPredictor`'s internal buffer
3. Prediction passes are triggered every `batch_size` messages
4. Results are summarised in a table

In [1]:
import sys
sys.path.insert(0, '..')

In [2]:
import logging
from collections import defaultdict
from pathlib import Path

import geopandas as gpd
import joblib
import pandas as pd
from shapely.geometry import Point

from src.ais_kafka_predictor import AISKafkaPredictor
from src.ais_stream_mock import AISStreamMock
from src.methods import dms_to_dd
from src.voyage_creator import VoyageCreator

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s — %(message)s')

## 1. Load port reference data

In [3]:
df_ports = pd.read_csv('../data/ports/ports.csv')
df_ports['lat_dd'] = df_ports['latitude'].apply(dms_to_dd)
df_ports['lon_dd'] = df_ports['longitude'].apply(dms_to_dd)
df_ports['geometry'] = df_ports.apply(lambda r: Point(r['lon_dd'], r['lat_dd']), axis=1)
gdf_ports = gpd.GeoDataFrame(df_ports, geometry='geometry', crs='EPSG:4326')
print(f'Loaded {len(gdf_ports)} ports')

Loaded 657 ports


## 2. Headless predictor (no Kafka)

`HeadlessPredictor` overrides `__init__` to skip the `KafkaConsumer` constructor
so the integration test runs without a broker. Everything else — buffer management,
feature engineering, model inference — is identical to the real class.

In [4]:
class HeadlessPredictor(AISKafkaPredictor):
    """AISKafkaPredictor with the Kafka consumer replaced by a simple list sink."""

    def __init__(self, ports_gdf, model_path='../models/delay_predictor.joblib', **kwargs):
        # Replicate __init__ without touching KafkaConsumer
        self.topic = 'headless'
        self.buffer_hours = kwargs.get('buffer_hours', 48.0)
        self.batch_size   = kwargs.get('batch_size', 500)
        self.timestamp_col = kwargs.get('timestamp_col', 'base_date_time')

        self._consumer = None  # not used
        self._model    = joblib.load(model_path)

        self._port_loc = ports_gdf.set_index('portName')[['lat_dd', 'lon_dd']]
        self._voyage_creator = VoyageCreator(
            ports_gdf,
            radius_nm=kwargs.get('voyage_radius_nm', 10),
            max_speed_knots=kwargs.get('voyage_max_speed', 1.5),
            gap_threshold_h=kwargs.get('voyage_gap_h', 24),
        )
        self._ping_buffer: dict = defaultdict(list)
        self.vessel_predictions: dict = {}

    def ingest(self, messages, verbose_every: int = 1000):
        """Feed an iterable of AIS dicts and trigger prediction passes automatically."""
        msg_count = 0
        for msg in messages:
            mmsi = msg.get('mmsi')
            if not mmsi:
                continue
            self._ping_buffer[int(mmsi)].append(msg)
            msg_count += 1
            if msg_count % self.batch_size == 0:
                self._run_prediction_pass()
                if msg_count % verbose_every == 0:
                    print(f'  {msg_count:,} messages ingested — '
                          f'{len(self.vessel_predictions)} vessel(s) with predictions')
        # Final pass on remaining messages
        self._run_prediction_pass()
        print(f'Done. Total messages: {msg_count:,} | '
              f'Vessels tracked: {len(self.vessel_predictions)}')
        return msg_count

## 3. Configure mock stream and predictor

In [5]:
# Use one week of data; speed_factor=1e12 makes the inter-message sleep negligible
PARQUET_FILES = sorted(Path('../data/ais').glob('ais-2025-01-0*.parquet'))[:7]
print('Parquet files to stream:')
for p in PARQUET_FILES:
    print(' ', p.name)

Parquet files to stream:
  ais-2025-01-01.parquet
  ais-2025-01-02.parquet
  ais-2025-01-03.parquet
  ais-2025-01-04.parquet
  ais-2025-01-05.parquet
  ais-2025-01-06.parquet
  ais-2025-01-07.parquet


In [6]:
predictor = HeadlessPredictor(
    ports_gdf=gdf_ports,
    model_path='../models/delay_predictor.joblib',
    buffer_hours=96,   # keep up to 4 days so multi-file voyages are detected
    batch_size=2000,   # trigger a prediction pass every 2 000 messages
    voyage_radius_nm=10,
    voyage_max_speed=1.5,
    voyage_gap_h=24,
)
print('HeadlessPredictor ready')

HeadlessPredictor ready


## 4. Stream all files through the predictor

`AISStreamMock` yields one dict per AIS record. We chain all files together and
pass the generator straight to `HeadlessPredictor.ingest()`.

In [ ]:
import itertools

def chain_mocks(parquet_files, speed_factor=1e12, timestamp_col='base_date_time'):
    """Yield messages from multiple Parquet files in sequence, sorted by timestamp within each."""
    for path in parquet_files:
        mock = AISStreamMock(path, speed_factor=speed_factor, timestamp_col=timestamp_col)
        yield from mock.stream()

total = predictor.ingest(
    chain_mocks(PARQUET_FILES),
    verbose_every=2000,
)

  2,000 messages ingested — 0 vessel(s) with predictions
  4,000 messages ingested — 0 vessel(s) with predictions
  6,000 messages ingested — 0 vessel(s) with predictions
  8,000 messages ingested — 0 vessel(s) with predictions
  10,000 messages ingested — 0 vessel(s) with predictions
  12,000 messages ingested — 0 vessel(s) with predictions
  14,000 messages ingested — 0 vessel(s) with predictions
  16,000 messages ingested — 0 vessel(s) with predictions
  18,000 messages ingested — 0 vessel(s) with predictions
  20,000 messages ingested — 0 vessel(s) with predictions
  22,000 messages ingested — 0 vessel(s) with predictions
  24,000 messages ingested — 0 vessel(s) with predictions
  26,000 messages ingested — 0 vessel(s) with predictions
  28,000 messages ingested — 0 vessel(s) with predictions
  30,000 messages ingested — 0 vessel(s) with predictions
  32,000 messages ingested — 0 vessel(s) with predictions
  34,000 messages ingested — 0 vessel(s) with predictions
  36,000 messages 

INFO src.ais_kafka_predictor — MMSI 563146800 | Gretna → Destrehan | 1.0 h remaining (predicted)
INFO src.ais_kafka_predictor — Prediction pass complete — 1 vessel(s) updated, 1 total tracked


  156,000 messages ingested — 1 vessel(s) with predictions


INFO src.ais_kafka_predictor — MMSI 563146800 | Gretna → Destrehan | 1.0 h remaining (predicted)
INFO src.ais_kafka_predictor — Prediction pass complete — 1 vessel(s) updated, 1 total tracked


  158,000 messages ingested — 1 vessel(s) with predictions


...

  906,000 messages ingested — 20 vessel(s) with predictions


INFO src.ais_kafka_predictor — MMSI 304679000 | Haymark Terminal → Houston | 1.7 h remaining (predicted)
INFO src.ais_kafka_predictor — MMSI 311008600 | Yerbabuena Island → Martinez | 3.1 h remaining (predicted)
INFO src.ais_kafka_predictor — MMSI 316206000 | Sault Ste Marie → Superior | 1.2 h remaining (predicted)
INFO src.ais_kafka_predictor — MMSI 338721000 | Long Beach → Port Hueneme | 5.1 h remaining (predicted)
INFO src.ais_kafka_predictor — MMSI 357329000 | New Orleans → St. James | -0.1 h remaining (predicted)
INFO src.ais_kafka_predictor — MMSI 366870320 | Miami → Port Everglades | -0.3 h remaining (predicted)
INFO src.ais_kafka_predictor — MMSI 366904930 | Sault Ste Marie → Presque Isle | -1.0 h remaining (predicted)
INFO src.ais_kafka_predictor — MMSI 366904940 | Monroe → Detroit | 0.3 h remaining (predicted)
INFO src.ais_kafka_predictor — MMSI 366938710 | Sault Ste Marie → Marquette | -0.0 h remaining (predicted)
INFO src.ais_kafka_predictor — MMSI 367313160 | Winslow → Eve

  908,000 messages ingested — 20 vessel(s) with predictions


INFO src.ais_kafka_predictor — MMSI 304679000 | Haymark Terminal → Deer Park | 2.6 h remaining (predicted)
INFO src.ais_kafka_predictor — MMSI 311008600 | Alameda → Mare Island | 2.4 h remaining (predicted)
INFO src.ais_kafka_predictor — MMSI 316206000 | Sault Ste Marie → Duluth | 3.2 h remaining (predicted)
INFO src.ais_kafka_predictor — MMSI 338721000 | Long Beach → Port Hueneme | 5.1 h remaining (predicted)
INFO src.ais_kafka_predictor — MMSI 357329000 | New Orleans → Convent | -0.3 h remaining (predicted)
INFO src.ais_kafka_predictor — MMSI 366870320 | Miami → Port Everglades | -0.3 h remaining (predicted)
INFO src.ais_kafka_predictor — MMSI 366904930 | Sault Ste Marie → Presque Isle | -1.0 h remaining (predicted)
INFO src.ais_kafka_predictor — MMSI 366904940 | Monroe → Detroit | 0.3 h remaining (predicted)
INFO src.ais_kafka_predictor — MMSI 366938710 | Sault Ste Marie → Presque Isle | -1.1 h remaining (predicted)
INFO src.ais_kafka_predictor — MMSI 367313160 | Creosote → Mukilteo

  910,000 messages ingested — 20 vessel(s) with predictions


## 5. Results

In [ ]:
summary = predictor.get_vessel_summary()
print(f'{len(summary)} vessel(s) with predictions')
summary

In [ ]:
if not summary.empty:
    print('predicted_remaining_hours — distribution:')
    print(summary['predicted_remaining_hours'].describe().round(2))
    print()
    print('Top routes by predicted remaining hours:')
    print(
        summary[['origin_port', 'destination_port', 'predicted_remaining_hours']]
        .sort_values('predicted_remaining_hours', ascending=False)
        .head(10)
        .to_string(index=False)
    )

## 6. Sanity checks

In [ ]:
assert total > 0, 'No messages were ingested'
assert len(predictor._ping_buffer) >= 0, 'Buffer should exist'

if not summary.empty:
    assert summary['predicted_remaining_hours'].notna().all(), \
        'All predictions should be non-null'
    assert (summary['predicted_remaining_hours'] >= 0).all(), \
        'Remaining hours should be non-negative'
    assert summary['origin_port'].notna().all(), \
        'Every prediction should have an origin port'
    assert summary['destination_port'].notna().all(), \
        'Every prediction should have a destination port'

print('All assertions passed.')